# 🌯 «КАПИБАРА-ШАУРМИСТ» — Сцена 1 «Открытие»

Генерация видео через **Wan2.1-T2V-1.3B**.

**Важно:** `Runtime → Change runtime type → T4 GPU` (бесплатный тариф подходит, нужно ~8.2 ГБ VRAM).

Порядок: просто выполнить все ячейки сверху вниз. Скачивание весов ~10 ГБ занимает несколько минут, генерация на T4 — примерно 15–20 минут.

🔧 Версия без `flash-attn`: используется встроенное внимание PyTorch (SDPA) — та же картинка, зато никакой 30-минутной компиляции, которая падает.

💡 **Если что-то не работает** — запусти ячейку 0 «Доктор» ещё раз: она покажет ✅/❌ по каждому компоненту. Красный текст из упавшей ячейки — пришли в чат.

In [39]:
# 0. «Доктор»: запускать ПЕРВЫМ и ПОСЛЕ любой ошибки — покажет, что сломано.
import os, shutil
print('=== Диагностика ===')

import torch
if torch.cuda.is_available():
    print('✅ GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('❌ GPU НЕТ. Runtime -> Change runtime type -> T4 GPU, потом запусти ячейки заново.')

repo = '/content/Wan2.1'
model_py = repo + '/wan/modules/model.py'
if os.path.isdir(repo + '/wan'):
    print('✅ Репозиторий на месте')
    src = open(model_py).read()
    if 'attention as flash_attention' in src:
        print('✅ Патч внимания применён (flash-attn не нужен)')
    elif 'from .attention import flash_attention' in src:
        print('❌ Патч внимания НЕ применён — запусти ячейку 1 ещё раз')
    else:
        print('⚠️  model.py изменился — напиши в чат')
else:
    print('❌ Репозиторий не найден — запусти ячейку 1')

ck = repo + '/checkpoints/Wan2.1-T2V-1.3B'
if os.path.isdir(ck):
    gb = sum(os.path.getsize(os.path.join(r, f)) for r, d, fs in os.walk(ck) for f in fs) / 1e9
    print(('✅ Веса на месте' if gb > 8 else '⚠️  Веса недокачаны') + f' ({gb:.1f} GB)')
else:
    print('❌ Веса не скачаны — запусти ячейку 2')

out = '/content/scene1_opening.mp4'
print(f'✅ Видео готово ({os.path.getsize(out)/1e6:.1f} MB)' if os.path.exists(out) else '⏳ Видео ещё нет — запусти ячейку 3')

free = shutil.disk_usage('/content').free / 1e9
print(('✅ Свободного диска хватит' if free > 15 else '❌ Мало места на диске') + f' ({free:.1f} GB)')

=== Диагностика ===
✅ GPU: Tesla T4 | VRAM: 15.6 GB
✅ Репозиторий на месте
✅ Патч внимания применён (flash-attn не нужен)
❌ Веса не скачаны — запусти ячейку 2
⏳ Видео ещё нет — запусти ячейку 3
✅ Свободного диска хватит (75.0 GB)


In [40]:
# 1. Достаём репозиторий и ставим зависимости (без flash-attn — он больше не нужен)
!([ -d Wan2.1 ] && echo '✅ Репозиторий уже тут') || git clone https://github.com/kuzyaleha228-oss/Wan2.1.git
%cd /content/Wan2.1
!git checkout arena/01a101c1-wan2-1 -q
!git stash -q || true
!git pull -q origin arena/01a101c1-wan2-1 || true

# Зависимости (строчку с flash_attn выкидываем)
!grep -v "^flash_attn" requirements.txt > requirements_no_fa.txt
!pip install -q -r requirements_no_fa.txt

# ПАТЧ: вместо падающего в сборке flash-attn — встроенное внимание PyTorch.
# Математически эквивалентно для генерации, просто чуть медленнее.
!sed -i 's/from .attention import flash_attention/from .attention import attention as flash_attention/' wan/modules/model.py

# Проверка патча
!grep -n 'flash_attention' wan/modules/model.py | head -1
print('✅ Ячейка 1 готова — можно запускать ячейку 2')

✅ Репозиторий уже тут
/content/Wan2.1
10:from .attention import attention as flash_attention
✅ Ячейка 1 готова — можно запускать ячейку 2


In [41]:
# 2. Скачиваем веса модели (~10 ГБ, один раз)
# ВАЖНО: хаб строго <2.0, иначе transformers ломается (уже было!)
!pip install -q "huggingface_hub[cli]>=1.5,<2.0"
!huggingface-cli download Wan-AI/Wan2.1-T2V-1.3B --local-dir ./checkpoints/Wan2.1-T2V-1.3B


Hint: A new version of huggingface_hub (2.1.1) is available! You are using version 1.33.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [42]:
# 3. ГЕНЕРАЦИЯ СЦЕНЫ 1 «Открытие» (832x480, 81 кадр, ~5 секунд)
%%time
!python generate.py \
  --task t2v-1.3B \
  --size "832*480" \
  --ckpt_dir ./checkpoints/Wan2.1-T2V-1.3B \
  --save_file /content/scene1_opening.mp4 \
  --prompt "A capybara wearing an apron and chef hat ceremonially opens the shutter of a small street food kiosk in Frankfurt, steam pouring out, sign reads CAPY SHAWARMA, morning light, cinematic, absurd comedy, high detail"

^C
CPU times: user 69.3 ms, sys: 10.4 ms, total: 79.7 ms
Wall time: 42.2 s


In [44]:
# 4. Проверяем файл и смотрим прямо в ноутбуке 🎬
import os
path = '/content/scene1_opening.mp4'
if not os.path.exists(path):
    raise SystemExit('❌ Видеофайл не найден — значит ячейка 3 (генерация) не завершилась. '
                     'Поднимись к ней, найди красный текст ошибки и пришли его мне.')
print(f'✅ Файл найден: {os.path.getsize(path)/1e6:.1f} MB')

from base64 import b64encode
from IPython.display import HTML
with open(path, 'rb') as f:
    data = b64encode(f.read()).decode()
HTML(f'<video controls width="720"><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

SystemExit: ❌ Видеофайл не найден — значит ячейка 3 (генерация) не завершилась. Поднимись к ней, найди красный текст ошибки и пришли его мне.

In [45]:
# 5. Скачать файл себе (если не сработал встроенный плеер выше)
from google.colab import files
files.download('/content/scene1_opening.mp4')

FileNotFoundError: Cannot find file: /content/scene1_opening.mp4

In [47]:
!pip install -q "huggingface_hub[cli]>=1.5,<2.0"
%cd /content/Wan2.1
!python generate.py --task t2v-1.3B --size "832*480" --ckpt_dir ./checkpoints/Wan2.1-T2V-1.3B --save_file /content/scene1_opening.mp4 --prompt "A capybara wearing an apron and chef hat ceremonially opens the shutter of a small street food kiosk in Frankfurt, steam pouring out, sign reads CAPY SHAWARMA, morning light, cinematic, absurd comedy, high detail"
import os
from base64 import b64encode
from IPython.display import HTML
print('Размер файла:', round(os.path.getsize('/content/scene1_opening.mp4')/1e6, 1), 'MB')
HTML(f'<video controls width="720"><source src="data:video/mp4;base64,{b64encode(open("/content/scene1_opening.mp4","rb").read()).decode()}" type="video/mp4"></video>')

/content/Wan2.1
^C


FileNotFoundError: [Errno 2] No such file or directory: '/content/scene1_opening.mp4'